# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset is described by a Croissant schema and contains annotated records including mass spectrometry protein data, peptide coverage, and abundances for extracellular vesicles isolated from human mast cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load whole dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
print(f"Title: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
List all available record sets and their corresponding field `@id`s so you can later reference them in extraction and analysis steps.

In [ ]:
# List all available record sets in the dataset with their @id
print("Available record sets (by @id):")
record_set_ids = []
for rs in dataset.metadata.record_set:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '')}")
    record_set_ids.append(rs['@id'])
# Let's display info about the fields of each record set
for rs in dataset.metadata.record_set:
    rs_id = rs['@id']
    print(f"\nRecord set: {rs_id}")
    print("Fields (@id | name | dataType):")
    for field in rs.get('field', []):
        print(f"  - {field['@id']} | {field.get('name', '')} | {field.get('dataType','')}")

## 3. Data Extraction
Let's load the main record set containing protein records and put it in a `pandas.DataFrame` for data analysis.

For this notebook, we'll use the primary record set whose `@id` contains the main protein data (for this dataset it's typically `'https://sen.science/doi/10.71728/senscience.vq4a-28xa/proteins'`, but check the actual value above).


In [ ]:
# Assign the protein record set @id (update if necessary after inspecting above)
protein_record_set_id = record_set_ids[0]  # If there's only one, it's likely the main one

# Load all records from the selected record set
protein_records = list(dataset.records(record_set=protein_record_set_id))

# Assemble DataFrame
protein_df = pd.DataFrame(protein_records)

print("Protein record set columns:")
print(protein_df.columns.tolist())
print("\nPreview (first 5 rows):")
protein_df.head()

## 4. Exploratory Data Analysis (EDA)
We will do some basic EDA steps:
- Select a numeric field, e.g., 'MW [kDa]' (molecular weight) or 'number of peptides'.
- Filter proteins with MW > 10 kDa or another interesting field.
- Normalize values.
- Optionally, group by another field, such as 'description' or other relevant categorical variable.

In [ ]:
# Pick a numeric field (adjust based on the columns in protein_df)
# For example, let's use 'MW [kDa]' if it exists, otherwise pick another, like 'number_of_peptides'
candidate_numeric_fields = ['MW [kDa]', 'MW_kDa', 'number of peptides', 'Peptides', 'unique_peptides', 'Abundance']
numeric_field = None
for field in candidate_numeric_fields:
    if field in protein_df.columns:
        numeric_field = field
        break
if numeric_field is None:
    raise ValueError("No suitable numeric field found!")
print(f"Selected numeric field for analysis: {numeric_field}")

# Keep only records with non-null, positive numeric_field
protein_df_valid = protein_df[protein_df[numeric_field].astype(float) > 0].copy()

threshold = 10
filtered_df = protein_df_valid[protein_df_valid[numeric_field].astype(float) > threshold]

print(f"Filtered records: {len(filtered_df)} with {numeric_field} > {threshold}")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field (z-score normalization)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()

print(f"\nNormalized {numeric_field}:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by 'description' or first string field
group_field = None
for col in protein_df.columns:
    if col != numeric_field and protein_df[col].dtype == object:
        group_field = col
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and (optionally) a scatter or bar plot for grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field].astype(float), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (> {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped_df exists, show top categories by mean
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=grouped_df.sort_values(numeric_field, ascending=False).head(20),
        y=group_field, x=numeric_field
    )
    plt.title(f"Top 20 {group_field} by mean {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, you loaded Mass Spectrometry data from a FAIR^2 dataset using `mlcroissant`, explored the schema and main record set by their `@id`, extracted the protein data, performed basic filtering, normalization, grouping, and visualized numeric distributions. This approach can be extended to other record sets and more advanced analyses as needed.